[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/divyamohan1993/dip-practical/blob/main/Practical_3.ipynb)

> **Run in Google Colab:** Click the badge above to open this notebook in Google Colab. The dataset downloads automatically — no setup needed.

<style>
/* =====================================================================
   Practical Handbook — Uniform Print Formatting (CSU2543 Digital Image Processing)
   Sections are color-coded: Aim (blue), Theory (purple), Code (green),
   Output (amber), Analysis (maroon). Code blocks are uniform across all
   practicals. Page layout: 1-inch margins, justified text, Times New Roman.
   ===================================================================== */
@page { margin: 1in; }
@media print {
  body, .jp-Notebook, .jupyter-renderer { background: white !important; }
  .jp-CodeMirrorEditor, .CodeMirror { font-size: 10pt !important; }
  .jp-Cell-inputCollapser, .jp-Cell-outputCollapser, .jp-Toolbar { display: none !important; }
  .dip-section-marker { page-break-inside: avoid; }
}

.jp-RenderedHTMLCommon, .jp-RenderedMarkdown {
  font-family: 'Times New Roman', Georgia, serif;
  font-size: 12pt;
  line-height: 1.55;
  text-align: justify;
}

/* Section markers — colored heading bars */
.dip-section-marker {
  display: block;
  font-weight: bold;
  font-size: 1.35em;
  padding: 0.45em 0.7em;
  margin: 1.1em 0 0.6em 0;
  border-left: 6px solid;
  border-radius: 3px;
  letter-spacing: 0.02em;
  page-break-after: avoid;
}
.dip-section-aim      { color: #003c8f; background: #e3f2fd; border-color: #1565c0; }
.dip-section-theory   { color: #4a148c; background: #f3e5f5; border-color: #6a1b9a; }
.dip-section-code     { color: #1b5e20; background: #e8f5e9; border-color: #2e7d32; }
.dip-section-output   { color: #e65100; background: #fff3e0; border-color: #ef6c00; }
.dip-section-analysis { color: #b71c1c; background: #ffebee; border-color: #c62828; }

/* Sub-section headings within a section ("Part 1", "Part 2", ...) */
.dip-subsection {
  font-weight: 600;
  font-size: 1.1em;
  margin: 0.9em 0 0.4em 0;
  color: #2e7d32;
  border-bottom: 1px solid #c8e6c9;
  padding-bottom: 0.15em;
  page-break-after: avoid;
}

/* Code cells — uniform JetBrains Mono / Source Code Pro across all practicals */
.jp-CodeMirrorEditor, .jp-CodeCell .jp-InputArea-editor,
.CodeMirror, pre, .highlight, code {
  font-family: 'JetBrains Mono', 'Source Code Pro', Consolas, 'Courier New', monospace !important;
}
.jp-CodeCell .jp-InputArea-editor, .CodeMirror, pre {
  font-size: 10pt !important;
  background: #f6f8fa !important;
  border-left: 3px solid #2e7d32 !important;
  border-radius: 0 !important;
  padding: 8px 12px !important;
}
code { background: #f6f8fa; padding: 1px 5px; border-radius: 2px; font-size: 0.95em; }

/* Output cells — amber tint, matching the Output section colour */
.jp-OutputArea-output {
  border-left: 3px solid #ef6c00 !important;
  background: #fffaf3 !important;
  padding: 6px 10px !important;
}

/* Analysis question lists */
.dip-analysis-list { padding-left: 1.4em; }
.dip-analysis-list li { margin-bottom: 0.5em; text-align: justify; }
</style>

# Practical 3: Image Negation, Subtraction, and Inversion

<span class="dip-section-marker dip-section-aim">1. Aim</span>

To implement and analyze fundamental intensity-transformation operations: image negation, image-pair subtraction, and intensity inversion, and to demonstrate their use in medical imaging applications such as digital subtraction angiography and shading correction.

<span class="dip-section-marker dip-section-theory">2. Description / Theory</span>

**Negation.** The image negative reverses the intensity scale via $s = (L-1) - r$ where $L$ is the number of intensity levels. It enhances visibility of detail in predominantly dark regions (e.g., bright lesions on a dark X-ray).

**Image subtraction.** The absolute difference $g(x,y) = \lvert f_1(x,y) - f_2(x,y) \rvert$ isolates structures that have changed between two frames. In **digital subtraction angiography** (DSA) a contrast-enhanced live image is subtracted from a baseline mask to reveal blood vessels; in **shading correction** a sensor-bias image is subtracted from the raw acquisition.

**Inversion** is identical to negation when applied to the difference image; it is useful when the difference is sparse and a dark-on-light rendering is preferred.

<span class="dip-section-marker dip-section-code">3. Code</span>

In [ ]:
# Install dependencies (uncomment for Google Colab)
# !pip install opencv-python-headless matplotlib numpy

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os


In [ ]:
# === AUTO-DOWNLOAD DATASET (works in Google Colab and locally) ===
import os, urllib.request, zipfile

# Choose which chapter to download (CH01-CH12)
CHAPTER = "CH02"  # Chapter 2: Digital Image Fundamentals
DATASET_PATH = f"datasets/{CHAPTER}/"
DOWNLOAD_BASE = "https://www.imageprocessingplace.com/downloads_V3/dip3e_downloads/dip3e_book_images"

if not os.path.exists(DATASET_PATH) or not any(f.endswith('.tif') for f in os.listdir(DATASET_PATH)):
    zip_name = f"DIP3E_{CHAPTER}_Original_Images.zip"
    url = f"{DOWNLOAD_BASE}/{zip_name}"
    print(f"Downloading {CHAPTER} dataset from imageprocessingplace.com...")
    urllib.request.urlretrieve(url, "chapter.zip")
    os.makedirs(DATASET_PATH, exist_ok=True)
    with zipfile.ZipFile("chapter.zip", "r") as z:
        for f in z.namelist():
            if f.lower().endswith(".tif"):
                fname = os.path.basename(f)
                if fname:
                    with z.open(f) as src, open(os.path.join(DATASET_PATH, fname), "wb") as dst:
                        dst.write(src.read())
    os.remove("chapter.zip")
    print(f"Downloaded {len([f for f in os.listdir(DATASET_PATH) if f.endswith('.tif')])} images")
else:
    print(f"Dataset ready: {len([f for f in os.listdir(DATASET_PATH) if f.endswith('.tif')])} images")

# List all available images
images = sorted([f for f in os.listdir(DATASET_PATH) if f.endswith('.tif')])
print(f"\nAvailable images ({len(images)}):")
for i, name in enumerate(images, 1):
    print(f"  {i}. {name}")

<span class="dip-subsection">Part 1: Grayscale Conversion</span>
Load an image and convert it to grayscale.


In [ ]:
# === SELECT YOUR IMAGE HERE ===
selected_image = "Fig0232(a)(partial_body_scan).tif"  # Change to any image

# Load the image
img_color = cv2.imread(os.path.join(DATASET_PATH, selected_image))
img_gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB))
ax1.set_title("Original (Color)")
ax1.axis('off')

ax2.imshow(img_gray, cmap='gray')
ax2.set_title("Grayscale")
ax2.axis('off')

plt.suptitle("Grayscale Conversion", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Color shape: {img_color.shape}, Grayscale shape: {img_gray.shape}")


<span class="dip-subsection">Part 2: Image Negation (s = 255 - r)</span>
Compute the negative of an image by inverting all pixel intensities.


In [ ]:
# === SELECT IMAGE FOR NEGATION ===
negation_image = "Fig0232(a)(partial_body_scan).tif"  # Change to any image

img = cv2.imread(os.path.join(DATASET_PATH, negation_image), cv2.IMREAD_GRAYSCALE)

# Compute negative: s = 255 - r
negative = 255 - img

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Original image
axes[0, 0].imshow(img, cmap='gray')
axes[0, 0].set_title("Original Image")
axes[0, 0].axis('off')

# Negative image
axes[0, 1].imshow(negative, cmap='gray')
axes[0, 1].set_title("Negative (s = 255 - r)")
axes[0, 1].axis('off')

# Original histogram
axes[1, 0].hist(img.ravel(), bins=256, range=(0, 256), color='black', alpha=0.7)
axes[1, 0].set_title("Original Histogram")
axes[1, 0].set_xlabel("Intensity")
axes[1, 0].set_ylabel("Frequency")

# Negative histogram
axes[1, 1].hist(negative.ravel(), bins=256, range=(0, 256), color='black', alpha=0.7)
axes[1, 1].set_title("Negative Histogram")
axes[1, 1].set_xlabel("Intensity")
axes[1, 1].set_ylabel("Frequency")

plt.suptitle(f"Image Negation: {negation_image}", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


<span class="dip-subsection">Part 3: Image Subtraction</span>
Compute the absolute difference between two related images to reveal hidden structures.

### 3a: Digital Subtraction Angiography


In [ ]:
# === ANGIOGRAPHY IMAGE PAIR (change to any two images) ===
mask_file = "Fig0228(a)(angiography_mask_image).tif"
live_file = "Fig0228(b)(angiography_live_ image).tif"

mask = cv2.imread(os.path.join(DATASET_PATH, mask_file), cv2.IMREAD_GRAYSCALE)
live = cv2.imread(os.path.join(DATASET_PATH, live_file), cv2.IMREAD_GRAYSCALE)

# Ensure same size
if mask.shape != live.shape:
    live = cv2.resize(live, (mask.shape[1], mask.shape[0]))

# Compute absolute difference
difference = cv2.absdiff(live, mask)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(mask, cmap='gray')
axes[0].set_title("Mask Image (Before Contrast)")
axes[0].axis('off')

axes[1].imshow(live, cmap='gray')
axes[1].set_title("Live Image (After Contrast)")
axes[1].axis('off')

axes[2].imshow(difference, cmap='gray')
axes[2].set_title("Subtraction: |Live - Mask|")
axes[2].axis('off')

plt.suptitle("Digital Subtraction Angiography", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 3b: Dental X-ray Subtraction


In [ ]:
# === DENTAL IMAGE PAIR (change to any two images) ===
dental_file = "Fig0230(a)(dental_xray).tif"
dental_mask_file = "Fig0230(b)(dental_xray_mask).tif"

dental = cv2.imread(os.path.join(DATASET_PATH, dental_file), cv2.IMREAD_GRAYSCALE)
dental_mask = cv2.imread(os.path.join(DATASET_PATH, dental_mask_file), cv2.IMREAD_GRAYSCALE)

if dental.shape != dental_mask.shape:
    dental_mask = cv2.resize(dental_mask, (dental.shape[1], dental.shape[0]))

dental_diff = cv2.absdiff(dental, dental_mask)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(dental, cmap='gray')
axes[0].set_title("Dental X-ray")
axes[0].axis('off')

axes[1].imshow(dental_mask, cmap='gray')
axes[1].set_title("Dental Mask")
axes[1].axis('off')

axes[2].imshow(dental_diff, cmap='gray')
axes[2].set_title("Subtraction: |Xray - Mask|")
axes[2].axis('off')

plt.suptitle("Dental X-ray Subtraction", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 3c: Shading Correction (Tungsten Filament)


In [ ]:
# === TUNGSTEN IMAGE PAIR (change to any two images) ===
filament_file = "Fig0229(a)(tungsten_filament_shaded).tif"
shading_file = "Fig0229(b)(tungsten_sensor_shading).tif"

filament = cv2.imread(os.path.join(DATASET_PATH, filament_file), cv2.IMREAD_GRAYSCALE)
shading = cv2.imread(os.path.join(DATASET_PATH, shading_file), cv2.IMREAD_GRAYSCALE)

if filament.shape != shading.shape:
    shading = cv2.resize(shading, (filament.shape[1], filament.shape[0]))

# Shading correction via subtraction
corrected = cv2.absdiff(filament, shading)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(filament, cmap='gray')
axes[0].set_title("Shaded Filament")
axes[0].axis('off')

axes[1].imshow(shading, cmap='gray')
axes[1].set_title("Sensor Shading Pattern")
axes[1].axis('off')

axes[2].imshow(corrected, cmap='gray')
axes[2].set_title("Corrected: |Filament - Shading|")
axes[2].axis('off')

plt.suptitle("Shading Correction", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


<span class="dip-subsection">Part 4: Inversion of Subtraction Result</span>
Apply intensity inversion to the subtraction result to enhance visibility.


In [ ]:
# Invert the angiography subtraction result
inverted_diff = 255 - difference

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(difference, cmap='gray')
axes[0].set_title("Subtraction Result")
axes[0].axis('off')

axes[1].imshow(inverted_diff, cmap='gray')
axes[1].set_title("Inverted Subtraction (255 - diff)")
axes[1].axis('off')

# Enhanced version with contrast stretching
enhanced = cv2.normalize(difference, None, 0, 255, cv2.NORM_MINMAX)
axes[2].imshow(enhanced, cmap='gray')
axes[2].set_title("Enhanced (Contrast Stretched)")
axes[2].axis('off')

plt.suptitle("Inversion and Enhancement Pipeline", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## Complete Pipeline Summary


In [ ]:
# Full pipeline on angiography images
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(mask, cmap='gray')
axes[0, 0].set_title("1. Mask Image")
axes[0, 0].axis('off')

axes[0, 1].imshow(live, cmap='gray')
axes[0, 1].set_title("2. Live Image")
axes[0, 1].axis('off')

axes[0, 2].imshow(difference, cmap='gray')
axes[0, 2].set_title("3. |Live - Mask|")
axes[0, 2].axis('off')

axes[1, 0].imshow(255 - mask, cmap='gray')
axes[1, 0].set_title("4. Negation of Mask")
axes[1, 0].axis('off')

axes[1, 1].imshow(inverted_diff, cmap='gray')
axes[1, 1].set_title("5. Inverted Difference")
axes[1, 1].axis('off')

axes[1, 2].imshow(enhanced, cmap='gray')
axes[1, 2].set_title("6. Enhanced Result")
axes[1, 2].axis('off')

plt.suptitle("Complete Pipeline: Subtraction -> Negation -> Enhancement", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


<span class="dip-section-marker dip-section-output">4. Output</span>

*All output (printed values, computed statistics, and rendered figures) appears immediately below the corresponding code cells when this notebook is executed top-to-bottom.*

<span class="dip-section-marker dip-section-analysis">5. Analysis / Conclusion</span>

1. In the angiography example, what structures become visible after subtraction that were not visible in either original image?

2. Why is absolute difference used instead of simple subtraction? What would happen with signed vs unsigned arithmetic?

3. Compare the histogram of the original image with its negative. What mathematical relationship exists between them?